<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 9 · Machine Learning Foundations and Workflow

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
Chapter 9 lays out the ML workflow. Here we:
- frame a next-day return prediction problem,
- build features/labels,
- run time-series splits with scikit-learn, and
- evaluate predictions with statistical and portfolio metrics.


### Getting Help While Building ML Pipelines
- **Chapter 3** for pandas basics.
- **Appendix B** for NumPy/pandas quick reference.
- **Appendix C** for scikit-learn API reminders.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"

from sklearn.linear_model import Ridge
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error

## 1. Problem Framing and Feature Set

We define our ML task as predicting next‑day AAPL returns from simple momentum and volatility features, building a clean DataFrame of predictors and labels.

In [ ]:
prices = pd.read_csv(
    DATA_PATH, parse_dates=["Date"]
).set_index("Date").sort_index().ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
feature_window = 20
momentum = prices["AAPL"].pct_change(feature_window)
volatility = log_rets["AAPL"].rolling(feature_window).std()
features = pd.concat(
    {"momentum": momentum, "volatility": volatility}, axis=1
).dropna()
labels = log_rets["AAPL"].shift(-1).rename("label")
frame = pd.concat([features, labels], axis=1).dropna()
frame.head()

### 1.2 Simple Performance Metrics Helper

To keep this notebook self‑contained, we define a small helper that
summarizes a return series with annualized return, volatility, Sharpe
ratio, and maximum drawdown.

In [ ]:
def performance_stats(returns: pd.Series, risk_free: float = 0.02) -> pd.Series:
    ann_ret = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe = (ann_ret - risk_free) / ann_vol if ann_vol > 0 else np.nan
    wealth = (1 + returns).cumprod()
    max_dd = (wealth / wealth.cummax() - 1).min()
    return pd.Series(
        {
            "annualized_return": ann_ret,
            "annualized_vol": ann_vol,
            "sharpe": sharpe,
            "max_drawdown": max_dd,
        }
    )

### 1.1 Train/Validation Split Helper

This helper wraps scikit‑learn's time‑series split logic so that each fold respects chronological order instead of shuffling observations.

In [ ]:
def rolling_split(X: pd.DataFrame, y: pd.Series, n_splits: int = 5):
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_idx, val_idx in tscv.split(X):
        yield X.iloc[train_idx], X.iloc[val_idx], y.iloc[train_idx], y.iloc[val_idx]

### 1.2 Feature Selection
We explicitly select the feature columns before fitting models.

In [ ]:
feature_cols = ['momentum', 'volatility']
X = frame[feature_cols]
y = frame['label']

## 2. Ridge Regression Baseline

We fit a ridge regression model as a transparent baseline, using cross‑validated folds to estimate out‑of‑sample error.

In [ ]:
def run_ridge_workflow(X: pd.DataFrame, y: pd.Series, alpha: float = 10.0):
    metrics = []
    preds = pd.Series(index=y.index, dtype=float)
    for X_train, X_val, y_train, y_val in rolling_split(X, y):
        model = Ridge(alpha=alpha)
        model.fit(X_train, y_train)
        fold_pred = model.predict(X_val)
        preds.loc[y_val.index] = fold_pred
        mse = mean_squared_error(y_val, fold_pred)
        metrics.append(mse)
    return preds.dropna(), np.mean(metrics)

predictions, avg_mse = run_ridge_workflow(frame[["momentum", "volatility"]],
frame["label"])
avg_mse

### 2.1 Information Coefficient and Portfolio Proxy

After training, we correlate predictions with realized returns and build a simple sign‑based strategy to connect statistical skill to portfolio performance.

In [ ]:
ic = predictions.corr(frame["label"].loc[predictions.index])
print(f"Information Coefficient: {ic:.3f}")
portfolio_returns = np.sign(predictions) * frame["label"].loc[predictions.index]
perf = performance_stats(portfolio_returns)
perf

## 3. Exercises
### Exercise 1 – Feature Expansion
Add rolling z-score and volume features, then rerun the ridge workflow.
<details><summary>Hint</summary>
Use <code>prices.rolling(window).mean()</code> and merge into <code>frame</code> before calling <code>run_ridge_workflow</code>.
</details>

### Exercise 2 – Classification Variant
Convert the label to a sign (+1/-1) and run logistic regression with the same time-series splits.
<details><summary>Hint</summary>
Use <code>sklearn.linear_model.LogisticRegression</code> with <code>solver='liblinear'</code>.
</details>

### Exercise 3 – Prediction Drift Monitor
Plot rolling correlation between predictions and realized returns to detect instability.
<details><summary>Hint</summary>
Use <code>predictions.rolling(126).corr(frame['label'])</code> and visualize via Matplotlib.
</details>


## 4. Takeaways for Chapter 9
- Splitting data chronologically prevents leakage.
- Even simple ridge models need careful validation and monitoring.
- Translating predictions into portfolio signals bridges ML metrics and investment outcomes.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">